<html><div style="font-size:7pt">This notebook may contain text, code and images generated by artificial intelligence. Used model: claude-sonnet-4-20250514, vision model: claude-sonnet-4-20250514, endpoint: None, bia-bob version: 0.34.3.. It is good scientific practice to check the code and results it produces carefully. <a href="https://github.com/haesleinhuepf/bia-bob">Read more about code generation using bia-bob</a></div></html>

# Image Embeddings and UMAP Generation

This notebook demonstrates how to:
1. Generate vision embeddings using the [openai/clip-vit-base-patch32](https://huggingface.co/openai/clip-vit-base-patch32)
2. Create a 3D UMAP visualization of the embeddings
3. Save the results for VEST

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from datasets import load_dataset
from umap import UMAP
import random
from pathlib import Path
import requests
from io import BytesIO
import torch

In [2]:
import transformers
transformers.__version__

'4.44.1'

In [3]:
base_dir = Path("./")
images_dir = base_dir / "images"

## Load Vision Embedding Model

In [4]:
from transformers import CLIPProcessor, CLIPModel
import pandas as pd
import torch
from PIL import Image
import requests
import os

# Initialize models
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

C:\Users\rober\miniconda3\envs\genai-gpu2\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## Generate Vision Embeddings for All Images

In [5]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F

# Get list of all image files in the images folder and subfolders
image_files = []
for root, dirs, files in os.walk(images_dir):
    for f in files:
        if f.lower().endswith('.png') or f.lower().endswith('.jpg'):
            image_files.append(os.path.join(root, f))

# Initialize lists to store results
filenames = []
embeddings = []
images = []

print(f"Processing {len(image_files)} images for embeddings...")

# Loop through all image files
for i, image_path in enumerate(image_files):
    # Obtain filename from path
    filename = os.path.relpath(image_path, images_dir)
    
    # Load the image
    current_image = Image.open(image_path)
    images.append(np.asarray(current_image))

    # Apply the processing pipeline
    current_inputs = clip_processor(images=current_image, return_tensors="pt")
    with torch.no_grad():
        current_np_emb = clip_model.get_image_features(**current_inputs).squeeze().tolist()

    # Store results
    filenames.append(filename)
    embeddings.append(current_np_emb)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(image_files)} images")



# Create DataFrame
df = pd.DataFrame({
    'filename': filenames,
    'embedding': embeddings
})

print(f"Successfully processed {len(df)} images")
display(df.head())

Processing 3563 images for embeddings...
Processed 100/3563 images
Processed 200/3563 images
Processed 300/3563 images
Processed 400/3563 images
Processed 500/3563 images
Processed 600/3563 images
Processed 700/3563 images
Processed 800/3563 images
Processed 900/3563 images
Processed 1000/3563 images
Processed 1100/3563 images
Processed 1200/3563 images
Processed 1300/3563 images
Processed 1400/3563 images
Processed 1500/3563 images
Processed 1600/3563 images
Processed 1700/3563 images
Processed 1800/3563 images
Processed 1900/3563 images
Processed 2000/3563 images
Processed 2100/3563 images
Processed 2200/3563 images
Processed 2300/3563 images
Processed 2400/3563 images
Processed 2500/3563 images
Processed 2600/3563 images
Processed 2700/3563 images
Processed 2800/3563 images
Processed 2900/3563 images
Processed 3000/3563 images
Processed 3100/3563 images
Processed 3200/3563 images
Processed 3300/3563 images
Processed 3400/3563 images
Processed 3500/3563 images
Successfully processed 

,filename,embedding
0,EM\Background\EM_10107.jpg,"[0.12380989640951157, -0.21389639377593994, 0...."
1,EM\Background\EM_10122.jpg,"[0.12578292191028595, -0.21270939707756042, 0...."
2,EM\Background\EM_11549_NW.jpg,"[0.21396933495998383, -0.28135332465171814, 0...."
3,EM\Background\EM_11652_NW.jpg,"[0.018749238923192024, -0.029139315709471703, ..."
4,EM\Background\EM_11652_SE.jpg,"[0.2543867230415344, -0.310322105884552, -0.06..."


## Create 3D UMAP Visualization from Embeddings

In [6]:
# Convert embeddings list to numpy array matrix
embedding_matrix = np.stack(df['embedding'].values)

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print("Creating 3D UMAP coordinates...")

# Create 3D UMAP
umap_reducer = UMAP(n_components=3, random_state=42)
umap_coords_actual = umap_reducer.fit_transform(embedding_matrix)

# Add UMAP coordinates to dataframe
df['x'] = umap_coords_actual[:, 0]
df['y'] = umap_coords_actual[:, 1]
df['z'] = umap_coords_actual[:, 2]

print("UMAP coordinates created successfully")
print(f"X range: {df['x'].min():.2f} to {df['x'].max():.2f}")
print(f"Y range: {df['y'].min():.2f} to {df['y'].max():.2f}")
print(f"Z range: {df['z'].min():.2f} to {df['z'].max():.2f}")
display(df.head())

Embedding matrix shape: (3563, 512)
Creating 3D UMAP coordinates...


C:\Users\rober\miniconda3\envs\genai-gpu2\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP coordinates created successfully
X range: -2.07 to 15.56
Y range: 4.49 to 8.51
Z range: 0.71 to 6.38


,filename,embedding,x,y,z
0,EM\Background\EM_10107.jpg,"[0.12380989640951157, -0.21389639377593994, 0....",14.222042,6.900047,5.142784
1,EM\Background\EM_10122.jpg,"[0.12578292191028595, -0.21270939707756042, 0....",14.082297,6.920494,5.238936
2,EM\Background\EM_11549_NW.jpg,"[0.21396933495998383, -0.28135332465171814, 0....",14.595526,7.018033,4.765746
3,EM\Background\EM_11652_NW.jpg,"[0.018749238923192024, -0.029139315709471703, ...",15.417882,6.684331,4.055713
4,EM\Background\EM_11652_SE.jpg,"[0.2543867230415344, -0.310322105884552, -0.06...",11.934289,6.822479,6.046017


## Save Results to CSV File

In [7]:
# Save the dataframe to CSV
output_path = base_dir / "data.csv"

# Convert embedding arrays to strings for CSV storage
df_to_save = df.copy()
df_to_save['embedding'] = df_to_save['embedding'].apply(lambda x: ','.join(map(str, x)))

df_to_save.to_csv(output_path, index=False)

print(f"Results saved to {output_path}")
print(f"Final dataframe shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df[['filename', 'x', 'y', 'z']].head(10))

Results saved to data.csv
Final dataframe shape: (3563, 5)
Columns: ['filename', 'embedding', 'x', 'y', 'z']


,filename,x,y,z
0,EM\Background\EM_10107.jpg,14.222042,6.900047,5.142784
1,EM\Background\EM_10122.jpg,14.082297,6.920494,5.238936
2,EM\Background\EM_11549_NW.jpg,14.595526,7.018033,4.765746
3,EM\Background\EM_11652_NW.jpg,15.417882,6.684331,4.055713
4,EM\Background\EM_11652_SE.jpg,11.934289,6.822479,6.046017
5,EM\Background\EM_1182.jpg,14.383684,6.600202,4.946671
6,EM\Background\EM_13070_NW.jpg,15.208268,6.391442,4.209393
7,EM\Background\EM_13070_SE.jpg,15.277817,6.412994,4.248121
8,EM\Background\EM_13216_NW.jpg,15.229865,6.476767,4.359120
9,EM\Background\EM_13216_SE.jpg,15.301679,6.447055,4.158406
